In [69]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from category_encoders import LeaveOneOutEncoder
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from tqdm.auto import tqdm

In [70]:
train = pd.read_excel('train.xlsx')
test = pd.read_excel('test.xlsx')
example = pd.read_csv('example.csv')
train.drop(train.columns[0], axis=1, inplace=True)
train.fillna(0, inplace=True)
train.drop('Статус брони', axis=1, inplace=True)

C:\Users\hedge\AppData\Local\Temp\ipykernel_24656\954808093.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0' has dtype incompatible with datetime64[ns], please explicitly cast to a compatible dtype first.
  train.fillna(0, inplace=True)


In [71]:
train['Дата отмены'] = pd.to_datetime(train['Дата отмены'])
train['Дата бронирования'] = pd.to_datetime(train['Дата бронирования'])
train = train[train["Стоимость"]>100]
train

,№ брони,Номеров,Стоимость,Внесена предоплата,Способ оплаты,Дата бронирования,Дата отмены,Заезд,Ночей,Выезд,Источник,Категория номера,Гостей,Гостиница
0,20230428-6634-194809261,1,25700.0,0,Внешняя система оплаты,2023-04-20 20:37:30,2023-04-20 20:39:15,2023-04-28 15:00:00,3,2023-05-01 12:00:00,Яндекс.Путешествия,Номер «Стандарт»,2,1
1,20220711-6634-144460018,1,24800.0,12400,Отложенная электронная оплата: Банк Россия (ба...,2022-06-18 14:17:02,1970-01-01 00:00:00,2022-07-11 15:00:00,2,2022-07-13 12:00:00,Официальный сайт,Номер «Стандарт»,2,1
2,20221204-16563-171020423,1,25800.0,12900,Банк. карта: Банк Россия (банк. карта),2022-11-14 22:59:30,1970-01-01 00:00:00,2022-12-04 15:00:00,2,2022-12-06 12:00:00,Официальный сайт,Номер «Студия»,2,4
3,20230918-7491-223512699,1,10500.0,0,Внешняя система оплаты (С предоплатой),2023-09-08 15:55:53,1970-01-01 00:00:00,2023-09-18 15:00:00,1,2023-09-19 12:00:00,Bronevik.com(new),Номер «Стандарт»,1,3
4,20230529-6634-200121971,1,28690.0,28690,Система быстрых платежей: Эквайринг ComfortBoo...,2023-05-20 19:54:13,1970-01-01 00:00:00,2023-05-29 15:00:00,2,2023-05-31 12:00:00,Официальный сайт,Номер «Люкс»,4,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26169,20230310-7492-177993190,1,18240.0,9120,Банк. карта: Банк Россия (банк. карта),2023-01-07 17:45:18,1970-01-01 00:00:00,2023-03-10 15:00:00,2,2023-03-12 12:00:00,Официальный сайт,Номер «Стандарт»,2,2
26170,20230625-16563-206126520,1,69600.0,23200,Банк. карта: Банк Россия (банк. карта),2023-06-20 17:54:17,1970-01-01 00:00:00,2023-06-25 15:00:00,3,2023-06-28 12:00:00,Официальный сайт,Номер «Студия»,3,4
26171,20220624-7492-137587082,1,55600.0,13900,Банк. карта: Банк Россия (банк. карта),2022-05-08 19:24:05,1970-01-01 00:00:00,2022-06-24 15:00:00,4,2022-06-28 12:00:00,Официальный сайт,Номер «Стандарт»,2,2
26172,20220427-7491-125459150,1,6300.0,0,Гарантия банковской картой,2022-02-19 09:55:50,2022-04-16 23:14:35,2022-04-27 15:00:00,1,2022-04-28 12:00:00,booking.com,Номер «Стандарт»,2,3


In [72]:
categorical_columns = ['Способ оплаты', 'Источник', 'Категория номера']
le = LabelEncoder()

In [73]:
label_encoders = {}
# for column in categorical_columns:
#     le = LabelEncoder()
#     train[column] = le.fit_transform(train[column])
#     label_encoders[column] = le
# train

In [74]:
# Создаем объект OneHotEncoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Преобразуем категориальные столбцы
encoded_data = encoder.fit_transform(train[categorical_columns])

# Преобразуем результат в DataFrame с правильными именами столбцов
encoded_columns = encoder.get_feature_names_out(categorical_columns)
encoded_df = pd.DataFrame(encoded_data, columns=encoded_columns)

# Соединяем обратно с исходным DataFrame (если нужно оставить другие столбцы)
train = train.drop(columns=categorical_columns)  # Удаляем старые категориальные столбцы
train = pd.concat([train.reset_index(drop=True), encoded_df], axis=1)

# Выводим результат
train

,№ брони,Номеров,Стоимость,Внесена предоплата,Дата бронирования,Дата отмены,Заезд,Ночей,Выезд,Гостей,...,Категория номера_1. Номер «Стандарт»\n2. Номер «Стандарт»\n3. Номер «Стандарт»\n4. Номер «Стандарт»\n5. Номер «Стандарт»\n6. Номер «Стандарт»,Категория номера_1. Номер «Стандарт»\n2. Номер «Стандарт»\n3. Номер «Стандарт»\n4. Номер «Стандарт» для маломобильных групп населения,Категория номера_1. Номер «Стандарт»\n2. Номер «Стандарт» для маломобильных групп населения,Категория номера_Апартаменты с 2 спальнями с отдельным входом,Категория номера_Коттедж с 2 спальнями,Категория номера_Коттедж с 3 спальнями,Категория номера_Номер «Люкс»,Категория номера_Номер «Стандарт»,Категория номера_Номер «Стандарт» для маломобильных групп населения,Категория номера_Номер «Студия»
0,20230428-6634-194809261,1,25700.0,0,2023-04-20 20:37:30,2023-04-20 20:39:15,2023-04-28 15:00:00,3,2023-05-01 12:00:00,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,20220711-6634-144460018,1,24800.0,12400,2022-06-18 14:17:02,1970-01-01 00:00:00,2022-07-11 15:00:00,2,2022-07-13 12:00:00,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,20221204-16563-171020423,1,25800.0,12900,2022-11-14 22:59:30,1970-01-01 00:00:00,2022-12-04 15:00:00,2,2022-12-06 12:00:00,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,20230918-7491-223512699,1,10500.0,0,2023-09-08 15:55:53,1970-01-01 00:00:00,2023-09-18 15:00:00,1,2023-09-19 12:00:00,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,20230529-6634-200121971,1,28690.0,28690,2023-05-20 19:54:13,1970-01-01 00:00:00,2023-05-29 15:00:00,2,2023-05-31 12:00:00,4,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26168,20230310-7492-177993190,1,18240.0,9120,2023-01-07 17:45:18,1970-01-01 00:00:00,2023-03-10 15:00:00,2,2023-03-12 12:00:00,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
26169,20230625-16563-206126520,1,69600.0,23200,2023-06-20 17:54:17,1970-01-01 00:00:00,2023-06-25 15:00:00,3,2023-06-28 12:00:00,3,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
26170,20220624-7492-137587082,1,55600.0,13900,2022-05-08 19:24:05,1970-01-01 00:00:00,2022-06-24 15:00:00,4,2022-06-28 12:00:00,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
26171,20220427-7491-125459150,1,6300.0,0,2022-02-19 09:55:50,2022-04-16 23:14:35,2022-04-27 15:00:00,1,2022-04-28 12:00:00,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [75]:
# Преобразование столбцов дат в тип datetime
train['Дата бронирования'] = pd.to_datetime(train['Дата бронирования'])
train['Заезд'] = pd.to_datetime(train['Заезд'])
train.head()

,№ брони,Номеров,Стоимость,Внесена предоплата,Дата бронирования,Дата отмены,Заезд,Ночей,Выезд,Гостей,...,Категория номера_1. Номер «Стандарт»\n2. Номер «Стандарт»\n3. Номер «Стандарт»\n4. Номер «Стандарт»\n5. Номер «Стандарт»\n6. Номер «Стандарт»,Категория номера_1. Номер «Стандарт»\n2. Номер «Стандарт»\n3. Номер «Стандарт»\n4. Номер «Стандарт» для маломобильных групп населения,Категория номера_1. Номер «Стандарт»\n2. Номер «Стандарт» для маломобильных групп населения,Категория номера_Апартаменты с 2 спальнями с отдельным входом,Категория номера_Коттедж с 2 спальнями,Категория номера_Коттедж с 3 спальнями,Категория номера_Номер «Люкс»,Категория номера_Номер «Стандарт»,Категория номера_Номер «Стандарт» для маломобильных групп населения,Категория номера_Номер «Студия»
0,20230428-6634-194809261,1,25700.0,0,2023-04-20 20:37:30,2023-04-20 20:39:15,2023-04-28 15:00:00,3,2023-05-01 12:00:00,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,20220711-6634-144460018,1,24800.0,12400,2022-06-18 14:17:02,1970-01-01 00:00:00,2022-07-11 15:00:00,2,2022-07-13 12:00:00,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,20221204-16563-171020423,1,25800.0,12900,2022-11-14 22:59:30,1970-01-01 00:00:00,2022-12-04 15:00:00,2,2022-12-06 12:00:00,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,20230918-7491-223512699,1,10500.0,0,2023-09-08 15:55:53,1970-01-01 00:00:00,2023-09-18 15:00:00,1,2023-09-19 12:00:00,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,20230529-6634-200121971,1,28690.0,28690,2023-05-20 19:54:13,1970-01-01 00:00:00,2023-05-29 15:00:00,2,2023-05-31 12:00:00,4,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [76]:
import holidays
# 
train["Разница_бронирование_заезд"] = (train['Заезд'] - train['Дата бронирования']).dt.days
train["Дней отпуска"] = (train['Выезд'] - train['Заезд']).dt.days
train["Сезон"] = train["Заезд"].dt.month
# 
train['День недели бронирования'] = train['Дата бронирования'].dt.weekday
train['День бронирования'] = train['Дата бронирования'].dt.day
train['День заезда'] = train['Заезд'].dt.day
train['Месяц заезда'] = train['Заезд'].dt.month
train['День недели заезда'] = train['Заезд'].dt.weekday

train['prepayment_ratio'] = train['Внесена предоплата'] / train['Стоимость']

# 1. Флаг полной предоплаты
train['Полная_предоплата'] = (train['Внесена предоплата'] >= train['Стоимость']).astype(int)

# 4. Флаг бронирования в выходной день
train['Бронирование_в_выходной'] = train['День недели бронирования'].apply(lambda x: 1 if x >= 5 else 0)

# 6. Бронирование в последний момент
train['Last_Minute_Booking'] = (train['Разница_бронирование_заезд'] <= 2).astype(int)

# 7. Длительное проживание
train['Длительное_проживание'] = (train['Ночей'] >= 7).astype(int)

# 8. Категории по "Разница_бронирование_заезд"
train['Категория_разницы_времени'] = pd.cut(train['Разница_бронирование_заезд'],
                                            bins=[-np.inf, 7, 30, 90, 180, np.inf],
                                            labels=False)

# 9. Категории по "Дней отпуска"
train['Категория_длительности_отпуска'] = pd.cut(train['Дней отпуска'],
                                                 bins=[-np.inf, 2, 5, 7, 14, np.inf],
                                                 labels=False)

# 10. Флаги праздничных дней (используя библиотеку holidays)
ru_holidays = holidays.Russia(years=[2022, 2023])  # Укажите нужные годы

train['Бронирование_в_праздник'] = train['Дата бронирования'].dt.date.apply(lambda x: 1 if x in ru_holidays else 0)
train['Заезд_в_праздник'] = train['Заезд'].dt.date.apply(lambda x: 1 if x in ru_holidays else 0)
# 
# 
# Функция для проверки пересечения интервалов
def check_date_overlap(start, end, event_start, event_end):
    return 1 if (start <= event_end and end >= event_start) else 0


train['пандемия_2022'] = train.apply(lambda x: check_date_overlap(x['Дата бронирования'], x['Заезд'],
                                                                  pd.Timestamp('2022-01-01'),
                                                                  pd.Timestamp('2022-02-28')), axis=1)

train['мобилизация_август_2022'] = train.apply(lambda x: check_date_overlap(x['Дата бронирования'], x['Заезд'],
                                                                          pd.Timestamp('2022-08-01'),
                                                                          pd.Timestamp('2022-11-30')), axis=1)

train['весенний_сезон_2023'] = train.apply(lambda x: check_date_overlap(x['Дата бронирования'], x['Заезд'],
                                                                        pd.Timestamp('2023-04-01'),
                                                                        pd.Timestamp('2023-06-30')), axis=1)

train['экономические_санкции_2022_2023'] = train.apply(lambda x: check_date_overlap(x['Дата бронирования'], x['Заезд'],
                                                                                    pd.Timestamp('2022-09-01'),
                                                                                    pd.Timestamp('2023-03-31')), axis=1)

train['военные_действия_февраль_2022'] = train.apply(lambda x: check_date_overlap(x['Дата бронирования'], x['Заезд'],
                                                                                pd.Timestamp('2022-02-01'),
                                                                                pd.Timestamp('2022-02-28')), axis=1)

train['крым_угрозы_май_2023'] = train.apply(lambda x: check_date_overlap(x['Дата бронирования'], x['Заезд'],
                                                                       pd.Timestamp('2023-05-01'),
                                                                       pd.Timestamp('2023-09-30')), axis=1)

# Добавляем колонку для периода с апреля по октябрь 2023 года
train['период_апрель_октябрь_2023'] = train.apply(lambda x: check_date_overlap(x['Дата бронирования'], x['Заезд'],
                                                                               pd.Timestamp('2023-04-01'),
                                                                               pd.Timestamp('2023-10-31')), axis=1)


In [77]:
usd_train = pd.read_csv("usd_train.csv")
usd_test = pd.read_csv("usd_test.csv")
train["usd_change_ab"]=usd_train['usd_rate_change_between_booking_and_arrival']
train['abs_usd']=abs(usd_train['usd_rate_at_arrival']-usd_train['usd_rate_at_booking'])
# train["usd_change_ab"].fillna(train["usd_change_ab"].mean(), inplace=True)
# train["abs_usd"].fillna(train["abs_usd"].mean(), inplace=True)


In [78]:
hotel_locations = {
    1: 'Yoshkar-Ola',
    2: 'Yoshkar-Ola',
    3: 'Volgograd',
    4: 'Volgograd'
}
train['City'] = train['Гостиница'].map(hotel_locations)
train['City'] = train['City'].str.capitalize()

train['Заезд'] = pd.to_datetime(train['Заезд'])
train['Выезд'] = pd.to_datetime(train['Выезд'])


In [79]:
unique_requests = pd.read_csv("unique_requests.csv")
unique_requests['Заезд'] = pd.to_datetime(unique_requests['Заезд'])
unique_requests['Выезд'] = pd.to_datetime(unique_requests['Выезд'])
# Объединяем данные с основным датасетом
train = train.merge(unique_requests, on=['City', 'Заезд', 'Выезд'], how='left')
train

,№ брони,Номеров,Стоимость,Внесена предоплата,Дата бронирования,Дата отмены,Заезд,Ночей,Выезд,Гостей,...,экономические_санкции_2022_2023,военные_действия_февраль_2022,крым_угрозы_май_2023,период_апрель_октябрь_2023,usd_change_ab,abs_usd,City,Unnamed: 0,Средняя_температура,Основное_погодное_условие
0,20230428-6634-194809261,1,25700.0,0,2023-04-20 20:37:30,2023-04-20 20:39:15,2023-04-28 15:00:00,3,2023-05-01 12:00:00,2,...,0,0,0,1,-0.116098,0.0948,Yoshkar-ola,0,12.666667,Легкая морось
1,20220711-6634-144460018,1,24800.0,12400,2022-06-18 14:17:02,1970-01-01 00:00:00,2022-07-11 15:00:00,2,2022-07-13 12:00:00,2,...,0,0,0,0,8.034371,4.5563,Yoshkar-ola,1,24.625000,Легкая морось
2,20221204-16563-171020423,1,25800.0,12900,2022-11-14 22:59:30,1970-01-01 00:00:00,2022-12-04 15:00:00,2,2022-12-06 12:00:00,2,...,1,0,0,0,2.585610,1.5570,Volgograd,2,-8.950000,Ясно
3,20230918-7491-223512699,1,10500.0,0,2023-09-08 15:55:53,1970-01-01 00:00:00,2023-09-18 15:00:00,1,2023-09-19 12:00:00,1,...,0,0,1,1,-1.591000,1.5623,Volgograd,3,17.100000,Переменная облачность
4,20230529-6634-200121971,1,28690.0,28690,2023-05-20 19:54:13,1970-01-01 00:00:00,2023-05-29 15:00:00,2,2023-05-31 12:00:00,4,...,0,0,1,1,0.071831,0.0574,Yoshkar-ola,4,15.650000,Облачно
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26168,20230310-7492-177993190,1,18240.0,9120,2023-01-07 17:45:18,1970-01-01 00:00:00,2023-03-10 15:00:00,2,2023-03-12 12:00:00,2,...,1,0,0,0,-0.001199,0.0011,Yoshkar-ola,2640,-3.150000,Сильный снегопад
26169,20230625-16563-206126520,1,69600.0,23200,2023-06-20 17:54:17,1970-01-01 00:00:00,2023-06-25 15:00:00,3,2023-06-28 12:00:00,3,...,0,0,1,1,7.912280,5.5653,Volgograd,3841,19.533333,Облачно
26170,20220624-7492-137587082,1,55600.0,13900,2022-05-08 19:24:05,1970-01-01 00:00:00,2022-06-24 15:00:00,4,2022-06-28 12:00:00,2,...,0,0,0,0,0.110375,0.0927,Yoshkar-ola,3658,18.925000,Ясно
26171,20220427-7491-125459150,1,6300.0,0,2022-02-19 09:55:50,2022-04-16 23:14:35,2022-04-27 15:00:00,1,2022-04-28 12:00:00,2,...,0,1,0,0,-20.815680,14.0265,Volgograd,1649,18.450000,Морось


In [80]:
train['Дата отмены'] = train['Дата отмены'].apply(lambda x: 1 if x != pd.Timestamp("1970-01-01 00:00:00") else 0)
train

,№ брони,Номеров,Стоимость,Внесена предоплата,Дата бронирования,Дата отмены,Заезд,Ночей,Выезд,Гостей,...,экономические_санкции_2022_2023,военные_действия_февраль_2022,крым_угрозы_май_2023,период_апрель_октябрь_2023,usd_change_ab,abs_usd,City,Unnamed: 0,Средняя_температура,Основное_погодное_условие
0,20230428-6634-194809261,1,25700.0,0,2023-04-20 20:37:30,1,2023-04-28 15:00:00,3,2023-05-01 12:00:00,2,...,0,0,0,1,-0.116098,0.0948,Yoshkar-ola,0,12.666667,Легкая морось
1,20220711-6634-144460018,1,24800.0,12400,2022-06-18 14:17:02,0,2022-07-11 15:00:00,2,2022-07-13 12:00:00,2,...,0,0,0,0,8.034371,4.5563,Yoshkar-ola,1,24.625000,Легкая морось
2,20221204-16563-171020423,1,25800.0,12900,2022-11-14 22:59:30,0,2022-12-04 15:00:00,2,2022-12-06 12:00:00,2,...,1,0,0,0,2.585610,1.5570,Volgograd,2,-8.950000,Ясно
3,20230918-7491-223512699,1,10500.0,0,2023-09-08 15:55:53,0,2023-09-18 15:00:00,1,2023-09-19 12:00:00,1,...,0,0,1,1,-1.591000,1.5623,Volgograd,3,17.100000,Переменная облачность
4,20230529-6634-200121971,1,28690.0,28690,2023-05-20 19:54:13,0,2023-05-29 15:00:00,2,2023-05-31 12:00:00,4,...,0,0,1,1,0.071831,0.0574,Yoshkar-ola,4,15.650000,Облачно
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26168,20230310-7492-177993190,1,18240.0,9120,2023-01-07 17:45:18,0,2023-03-10 15:00:00,2,2023-03-12 12:00:00,2,...,1,0,0,0,-0.001199,0.0011,Yoshkar-ola,2640,-3.150000,Сильный снегопад
26169,20230625-16563-206126520,1,69600.0,23200,2023-06-20 17:54:17,0,2023-06-25 15:00:00,3,2023-06-28 12:00:00,3,...,0,0,1,1,7.912280,5.5653,Volgograd,3841,19.533333,Облачно
26170,20220624-7492-137587082,1,55600.0,13900,2022-05-08 19:24:05,0,2022-06-24 15:00:00,4,2022-06-28 12:00:00,2,...,0,0,0,0,0.110375,0.0927,Yoshkar-ola,3658,18.925000,Ясно
26171,20220427-7491-125459150,1,6300.0,0,2022-02-19 09:55:50,1,2022-04-27 15:00:00,1,2022-04-28 12:00:00,2,...,0,1,0,0,-20.815680,14.0265,Volgograd,1649,18.450000,Морось


In [81]:
train = train[train["Стоимость"] > train["Внесена предоплата"]]


X = train.drop(["№ брони", "Дата бронирования", "Выезд", "Заезд", "Дата отмены"], axis=1)
y = train["Дата отмены"]
X

,Номеров,Стоимость,Внесена предоплата,Ночей,Гостей,Гостиница,Способ оплаты_Банк. карта (SberPay): Эквайринг ComfortBooking (Банк. карта) (SberPay),Способ оплаты_Банк. карта (Yandex Pay): Эквайринг ComfortBooking (Банк. карта) (Yandex Pay),Способ оплаты_Банк. карта [Кешбэк. МИР]: Эквайринг ComfortBooking (Банк. карта),Способ оплаты_Банк. карта [Кешбэк. МИР]: Эквайринг TravelLine Pro (Банк. карта),...,экономические_санкции_2022_2023,военные_действия_февраль_2022,крым_угрозы_май_2023,период_апрель_октябрь_2023,usd_change_ab,abs_usd,City,Unnamed: 0,Средняя_температура,Основное_погодное_условие
0,1,25700.0,0,3,2,1,0.0,0.0,0.0,0.0,...,0,0,0,1,-0.116098,0.0948,Yoshkar-ola,0,12.666667,Легкая морось
1,1,24800.0,12400,2,2,1,0.0,0.0,0.0,0.0,...,0,0,0,0,8.034371,4.5563,Yoshkar-ola,1,24.625000,Легкая морось
2,1,25800.0,12900,2,2,4,0.0,0.0,0.0,0.0,...,1,0,0,0,2.585610,1.5570,Volgograd,2,-8.950000,Ясно
3,1,10500.0,0,1,1,3,0.0,0.0,0.0,0.0,...,0,0,1,1,-1.591000,1.5623,Volgograd,3,17.100000,Переменная облачность
5,1,39100.0,8755,4,3,3,0.0,0.0,0.0,0.0,...,0,0,1,1,-4.650440,4.6988,Volgograd,5,20.087500,Преимущественно ясно
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26165,1,10300.0,0,1,2,3,0.0,0.0,0.0,0.0,...,0,0,1,1,-8.150461,5.8924,Volgograd,4005,19.050000,Преимущественно ясно
26168,1,18240.0,9120,2,2,2,0.0,0.0,0.0,0.0,...,1,0,0,0,-0.001199,0.0011,Yoshkar-ola,2640,-3.150000,Сильный снегопад
26169,1,69600.0,23200,3,3,4,0.0,0.0,0.0,0.0,...,0,0,1,1,7.912280,5.5653,Volgograd,3841,19.533333,Облачно
26170,1,55600.0,13900,4,2,2,0.0,0.0,0.0,0.0,...,0,0,0,0,0.110375,0.0927,Yoshkar-ola,3658,18.925000,Ясно


WEATHER

In [82]:
le = LabelEncoder()
X["Основное_погодное_условие"] = le.fit_transform(X["Основное_погодное_условие"])
X.drop(["Unnamed: 0","City"],axis=1,inplace=True)
X

,Номеров,Стоимость,Внесена предоплата,Ночей,Гостей,Гостиница,Способ оплаты_Банк. карта (SberPay): Эквайринг ComfortBooking (Банк. карта) (SberPay),Способ оплаты_Банк. карта (Yandex Pay): Эквайринг ComfortBooking (Банк. карта) (Yandex Pay),Способ оплаты_Банк. карта [Кешбэк. МИР]: Эквайринг ComfortBooking (Банк. карта),Способ оплаты_Банк. карта [Кешбэк. МИР]: Эквайринг TravelLine Pro (Банк. карта),...,мобилизация_август_2022,весенний_сезон_2023,экономические_санкции_2022_2023,военные_действия_февраль_2022,крым_угрозы_май_2023,период_апрель_октябрь_2023,usd_change_ab,abs_usd,Средняя_температура,Основное_погодное_условие
0,1,25700.0,0,3,2,1,0.0,0.0,0.0,0.0,...,0,1,0,0,0,1,-0.116098,0.0948,12.666667,1
1,1,24800.0,12400,2,2,1,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,8.034371,4.5563,24.625000,1
2,1,25800.0,12900,2,2,4,0.0,0.0,0.0,0.0,...,1,0,1,0,0,0,2.585610,1.5570,-8.950000,12
3,1,10500.0,0,1,1,3,0.0,0.0,0.0,0.0,...,0,0,0,0,1,1,-1.591000,1.5623,17.100000,6
5,1,39100.0,8755,4,3,3,0.0,0.0,0.0,0.0,...,0,0,0,0,1,1,-4.650440,4.6988,20.087500,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26165,1,10300.0,0,1,2,3,0.0,0.0,0.0,0.0,...,0,0,0,0,1,1,-8.150461,5.8924,19.050000,7
26168,1,18240.0,9120,2,2,2,0.0,0.0,0.0,0.0,...,0,0,1,0,0,0,-0.001199,0.0011,-3.150000,10
26169,1,69600.0,23200,3,3,4,0.0,0.0,0.0,0.0,...,0,1,0,0,1,1,7.912280,5.5653,19.533333,5
26170,1,55600.0,13900,4,2,2,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0.110375,0.0927,18.925000,12


In [83]:
#X = X.fillna(0)
# X.drop("Категория_времени_заезда",axis=1,inplace=True)

TEST DATA PREPARE

In [84]:
categorical_columns = ['Способ оплаты', 'Источник', 'Категория номера']
label_encoders = {}
# for column in categorical_columns:
#     le = LabelEncoder()
#     test[column] = le.fit_transform(test[column])
#     label_encoders[column] = le

# Преобразуем тестовый набор данных (используем обученный encoder)
encoded_test = encoder.transform(test[categorical_columns])

# Преобразуем результат в DataFrame с правильными именами столбцов
encoded_test_df = pd.DataFrame(encoded_test, columns=encoded_columns)

# Соединяем обратно с тестовым DataFrame (удаляем старые категориальные столбцы)
test = test.drop(columns=categorical_columns)
test = pd.concat([test.reset_index(drop=True), encoded_test_df], axis=1)

import holidays
# 
test["Разница_бронирование_заезд"] = (test['Заезд'] - test['Дата бронирования']).dt.days
test["Дней отпуска"] = (test['Выезд'] - test['Заезд']).dt.days
test["Сезон"] = test["Заезд"].dt.month
# 
test['День недели бронирования'] = test['Дата бронирования'].dt.weekday
test['День бронирования'] = test['Дата бронирования'].dt.day
test['День заезда'] = test['Заезд'].dt.day
test['Месяц заезда'] = test['Заезд'].dt.month
test['День недели заезда'] = test['Заезд'].dt.weekday

test['prepayment_ratio'] = test['Внесена предоплата'] / test['Стоимость']

# 1. Флаг полной предоплаты
test['Полная_предоплата'] = (test['Внесена предоплата'] >= test['Стоимость']).astype(int)

# 4. Флаг бронирования в выходной день
test['Бронирование_в_выходной'] = test['День недели бронирования'].apply(lambda x: 1 if x >= 5 else 0)

# 6. Бронирование в последний момент
test['Last_Minute_Booking'] = (test['Разница_бронирование_заезд'] <= 2).astype(int)

# 7. Длительное проживание
test['Длительное_проживание'] = (test['Ночей'] >= 7).astype(int)

# 8. Категории по "Разница_бронирование_заезд"
test['Категория_разницы_времени'] = pd.cut(test['Разница_бронирование_заезд'],
                                            bins=[-np.inf, 7, 30, 90, 180, np.inf],
                                            labels=False)

# 9. Категории по "Дней отпуска"
test['Категория_длительности_отпуска'] = pd.cut(test['Дней отпуска'],
                                                 bins=[-np.inf, 2, 5, 7, 14, np.inf],
                                                 labels=False)

# 10. Флаги праздничных дней (используя библиотеку holidays)
ru_holidays = holidays.Russia(years=[2022, 2023])  # Укажите нужные годы

test['Бронирование_в_праздник'] = test['Дата бронирования'].dt.date.apply(lambda x: 1 if x in ru_holidays else 0)
test['Заезд_в_праздник'] = test['Заезд'].dt.date.apply(lambda x: 1 if x in ru_holidays else 0)
# 
# 
# Функция для проверки пересечения интервалов
def check_date_overlap(start, end, event_start, event_end):
    return 1 if (start <= event_end and end >= event_start) else 0


test['пандемия_2022'] = test.apply(lambda x: check_date_overlap(x['Дата бронирования'], x['Заезд'],
                                                                  pd.Timestamp('2022-01-01'),
                                                                  pd.Timestamp('2022-02-28')), axis=1)

test['мобилизация_август_2022'] = test.apply(lambda x: check_date_overlap(x['Дата бронирования'], x['Заезд'],
                                                                            pd.Timestamp('2022-08-01'),
                                                                            pd.Timestamp('2022-11-30')), axis=1)

test['весенний_сезон_2023'] = test.apply(lambda x: check_date_overlap(x['Дата бронирования'], x['Заезд'],
                                                                        pd.Timestamp('2023-04-01'),
                                                                        pd.Timestamp('2023-06-30')), axis=1)

test['экономические_санкции_2022_2023'] = test.apply(lambda x: check_date_overlap(x['Дата бронирования'], x['Заезд'],
                                                                                    pd.Timestamp('2022-09-01'),
                                                                                    pd.Timestamp('2023-03-31')), axis=1)

test['военные_действия_февраль_2022'] = test.apply(lambda x: check_date_overlap(x['Дата бронирования'], x['Заезд'],
                                                                                  pd.Timestamp('2022-02-01'),
                                                                                  pd.Timestamp('2022-02-28')), axis=1)

test['крым_угрозы_май_2023'] = test.apply(lambda x: check_date_overlap(x['Дата бронирования'], x['Заезд'],
                                                                         pd.Timestamp('2023-05-01'),
                                                                         pd.Timestamp('2023-09-30')), axis=1)

# Добавляем колонку для периода с апреля по октябрь 2023 года
test['период_апрель_октябрь_2023'] = test.apply(lambda x: check_date_overlap(x['Дата бронирования'], x['Заезд'],
                                                                               pd.Timestamp('2023-04-01'),
                                                                               pd.Timestamp('2023-10-31')), axis=1)
test.drop(["Unnamed: 0"], axis=1, inplace=True)


test["usd_change_ab"]=usd_test['usd_rate_change_between_booking_and_arrival']
test['abs_usd']=abs(usd_test['usd_rate_at_arrival']-usd_test['usd_rate_at_booking'])
# test["usd_change_ab"].fillna(test["usd_change_ab"].mean(), inplace=True)
# test["abs_usd"].fillna(test["abs_usd"].mean(), inplace=True)

hotel_locations = {
    1: 'Yoshkar-Ola',
    2: 'Yoshkar-Ola',
    3: 'Volgograd',
    4: 'Volgograd'
}
test['City'] = test['Гостиница'].map(hotel_locations)
test['City'] = test['City'].str.capitalize()
# 
test['Заезд'] = pd.to_datetime(test['Заезд'])
test['Выезд'] = pd.to_datetime(test['Выезд'])
# 
unique_requests = pd.read_csv("unique_requests.csv")
unique_requests['Заезд'] = pd.to_datetime(unique_requests['Заезд'])
unique_requests['Выезд'] = pd.to_datetime(unique_requests['Выезд'])
test = test.merge(unique_requests, on=['City', 'Заезд', 'Выезд'], how='left')
# 
test.drop(["№ брони", "Дата бронирования", "Выезд", "Заезд", 'Unnamed: 0', "City"], axis=1, inplace=True)
test

,Номеров,Стоимость,Внесена предоплата,Ночей,Гостей,Гостиница,Способ оплаты_Банк. карта (SberPay): Эквайринг ComfortBooking (Банк. карта) (SberPay),Способ оплаты_Банк. карта (Yandex Pay): Эквайринг ComfortBooking (Банк. карта) (Yandex Pay),Способ оплаты_Банк. карта [Кешбэк. МИР]: Эквайринг ComfortBooking (Банк. карта),Способ оплаты_Банк. карта [Кешбэк. МИР]: Эквайринг TravelLine Pro (Банк. карта),...,мобилизация_август_2022,весенний_сезон_2023,экономические_санкции_2022_2023,военные_действия_февраль_2022,крым_угрозы_май_2023,период_апрель_октябрь_2023,usd_change_ab,abs_usd,Средняя_температура,Основное_погодное_условие
0,1,23750.0,23750,2,3,4,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,-0.106308,0.0943,1.700000,Легкий дождь
1,1,15010.0,7505,2,2,3,0.0,0.0,0.0,0.0,...,0,0,1,0,0,0,3.569179,2.2265,-2.725000,Облачно
2,1,8400.0,8400,1,2,1,0.0,0.0,0.0,0.0,...,1,0,1,0,0,0,3.303216,1.9947,-1.700000,Сильный снегопад
3,1,42500.0,42500,3,4,1,0.0,0.0,0.0,0.0,...,0,0,0,0,1,1,3.298987,2.9830,14.450000,Облачно
4,1,62500.0,11900,5,1,1,0.0,0.0,0.0,0.0,...,0,0,1,0,0,0,-0.660377,0.5082,5.000000,Облачно
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11213,1,50200.0,50200,2,4,2,0.0,0.0,0.0,1.0,...,0,0,0,0,0,0,-29.559866,28.2775,10.075000,Переменная облачность
11214,1,190100.0,43500,5,4,1,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,-0.249637,0.2298,NaN,NaN
11215,1,42300.0,42300,1,5,1,0.0,0.0,0.0,0.0,...,0,0,0,1,0,0,0.000000,0.0000,-2.750000,Снегопад
11216,1,27900.0,27900,1,4,1,0.0,0.0,0.0,0.0,...,0,0,1,0,0,0,0.978073,0.6716,-5.050000,Легкий снегопад


In [85]:
test["Средняя_температура"].fillna(test["Средняя_температура"].mean(), inplace=True)
test["Основное_погодное_условие"].fillna("unknown", inplace=True)
le = LabelEncoder()
test["Основное_погодное_условие"] = le.fit_transform(test["Основное_погодное_условие"])
test

C:\Users\hedge\AppData\Local\Temp\ipykernel_24656\521632137.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test["Средняя_температура"].fillna(test["Средняя_температура"].mean(), inplace=True)
C:\Users\hedge\AppData\Local\Temp\ipykernel_24656\521632137.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values 

,Номеров,Стоимость,Внесена предоплата,Ночей,Гостей,Гостиница,Способ оплаты_Банк. карта (SberPay): Эквайринг ComfortBooking (Банк. карта) (SberPay),Способ оплаты_Банк. карта (Yandex Pay): Эквайринг ComfortBooking (Банк. карта) (Yandex Pay),Способ оплаты_Банк. карта [Кешбэк. МИР]: Эквайринг ComfortBooking (Банк. карта),Способ оплаты_Банк. карта [Кешбэк. МИР]: Эквайринг TravelLine Pro (Банк. карта),...,мобилизация_август_2022,весенний_сезон_2023,экономические_санкции_2022_2023,военные_действия_февраль_2022,крым_угрозы_май_2023,период_апрель_октябрь_2023,usd_change_ab,abs_usd,Средняя_температура,Основное_погодное_условие
0,1,23750.0,23750,2,3,4,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,-0.106308,0.0943,1.700000,3
1,1,15010.0,7505,2,2,3,0.0,0.0,0.0,0.0,...,0,0,1,0,0,0,3.569179,2.2265,-2.725000,6
2,1,8400.0,8400,1,2,1,0.0,0.0,0.0,0.0,...,1,0,1,0,0,0,3.303216,1.9947,-1.700000,11
3,1,42500.0,42500,3,4,1,0.0,0.0,0.0,0.0,...,0,0,0,0,1,1,3.298987,2.9830,14.450000,6
4,1,62500.0,11900,5,1,1,0.0,0.0,0.0,0.0,...,0,0,1,0,0,0,-0.660377,0.5082,5.000000,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11213,1,50200.0,50200,2,4,2,0.0,0.0,0.0,1.0,...,0,0,0,0,0,0,-29.559866,28.2775,10.075000,7
11214,1,190100.0,43500,5,4,1,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,-0.249637,0.2298,8.444793,0
11215,1,42300.0,42300,1,5,1,0.0,0.0,0.0,0.0,...,0,0,0,1,0,0,0.000000,0.0000,-2.750000,12
11216,1,27900.0,27900,1,4,1,0.0,0.0,0.0,0.0,...,0,0,1,0,0,0,0.978073,0.6716,-5.050000,4


RANDOM FOREST

In [86]:
X["Стоимость"] = X['Стоимость'].apply(lambda x: np.log(x) if x > 0 else x)
X["Внесена предоплата"] = X['Внесена предоплата'].apply(lambda x: np.log(x) if x > 0 else x)

test["Стоимость"] = test['Стоимость'].apply(lambda x: np.log(x) if x > 0 else x)
test["Внесена предоплата"] = test['Внесена предоплата'].apply(lambda x: np.log(x) if x > 0 else x)


In [87]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold

In [88]:
n_splits = 10
skf = StratifiedKFold(n_splits=10, random_state=42, shuffle = True)

global_score = []
y_pred_global = []
# X = X.drop(["Средняя_температура","Основное_погодное_условие"],axis=1)
# test = test.drop(["Средняя_температура","Основное_погодное_условие"],axis=1)
for fold, (train_index, val_index) in enumerate(skf.split(X, y)):
    print(f'Фолд {fold + 1}/{n_splits}')
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    model = RandomForestClassifier(n_estimators=500, random_state=42, bootstrap=True, n_jobs=23, min_samples_leaf=4,
                                   min_samples_split=4, max_features="sqrt")
    model.fit(X_train, y_train)

    preds = model.predict_proba(X_val)[:, 1]
    score = roc_auc_score(y_val, preds)
    print(score)
    global_score.append(score)

    y_pred_global.append(model.predict_proba(test)[:, 1])

np.mean(global_score)

Фолд 1/10
0.886288482199573
Фолд 2/10
0.9049187021661973
Фолд 3/10
0.8606707987394386
Фолд 4/10
0.8917686597635924
Фолд 5/10
0.8852237391768659
Фолд 6/10
0.890588192183263
Фолд 7/10
0.8892847046998837
Фолд 8/10
0.8917758899888375
Фолд 9/10
0.8710963907827919
Фолд 10/10
0.8867933674672592


0.8858408927167701

In [34]:
X.to_csv("X_last.csv", index=False)

In [89]:
np.mean(y_pred_global, axis=0)

array([0.05037879, 0.03021374, 0.32960921, ..., 0.08464616, 0.08365875,
       0.22440314])

In [90]:
pd.DataFrame(np.mean(y_pred_global, axis=0)).to_csv("submit_rf_3.csv", index=False, header=False)


In [96]:
X.to_csv("X.csv", index=False)
y.to_csv("y.csv", index=False)
test.to_csv("test.csv", index=False)

In [95]:

# Получаем важность признаков
importances = model.feature_importances_

# Создаем DataFrame для удобства отображения
feature_names = X_val.columns
feature_importance_df = pd.DataFrame({'Признак': feature_names, 'Важность': importances})

# Сортируем признаки по важности
feature_importance_df = feature_importance_df.sort_values(by='Важность', ascending=False)

# Отображаем важность признаков
feature_importance_df

,Признак,Важность
83,prepayment_ratio,0.155633
17,Способ оплаты_Отложенная электронная оплата: Б...,0.146606
2,Внесена предоплата,0.143132
75,Разница_бронирование_заезд,0.050929
10,Способ оплаты_Банк. карта: Банк Россия (банк. ...,0.036873
...,...,...
65,Категория номера_1. Номер «Стандарт»\n2. Номер...,0.000000
66,Категория номера_1. Номер «Стандарт»\n2. Номер...,0.000000
67,Категория номера_1. Номер «Стандарт»\n2. Номер...,0.000000
73,Категория номера_Номер «Стандарт» для маломоби...,0.000000
